# CLV×상품 최소 상호작용 빠른 실험 — Dunnhumby

M1의 이진 구매 그래프를 그대로 두고, `LightGCN(u,i) + centered_CLV(u) × a_i`만 동일 BPR 손실로 공동 학습합니다. 실제 CLV와 N 등작량을 보존한 CLV-shuffle을 seed 42, 100 epoch에서 비교합니다. 최종 test·holdout은 생성하지 않는 사후적 과거 개발구간 탐색입니다.

## 1. 검토된 코드와 Drive 준비

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')
REVIEWED_SHA = '1a36c0e7423f69e13a57bd99085b674d32bfa73c'
REPO = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVIEWED_SHA], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert head == REVIEWED_SHA, (head, REVIEWED_SHA)
os.chdir(REPO)
print('reviewed source:', head)

## 2. 실행 조건 확인

In [ ]:
import json, torch
from lightgcn_clv_item_interaction import configure_clv_item_interaction_run, preflight_summary

assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
cfg = configure_clv_item_interaction_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

## 3. M1·CLV·CLV-shuffle 비교

M1은 동일 설정의 기존 결과를 재사용하고, CLV 두 arm만 학습합니다. 중단되어도 완료 epoch 다음부터 재개됩니다.

In [ ]:
from lightgcn_clv_item_interaction import run_clv_item_interaction

result = run_clv_item_interaction(cfg)
display(result)
print('\n균형 지표 판독:')
print(json.dumps(result.attrs['decision'], ensure_ascii=False, indent=2))
print('\n결과 파일:')
print(json.dumps(result.attrs['result_paths'], ensure_ascii=False, indent=2))